## **IMGW Weather Archive: Automated Data Collection Script**

### **Introductory information**

The notebook contains a script for automatic download of archived hydrological and meteorological data provided by the Institute of Meteorology and Water Management - National Research Institute (IMGW-PIB). This institution provides meteorological and hydrological services on the territory of the Republic of Poland. Before working with this script, please read the full information about the source of the data and the rules for its use:

* https://danepubliczne.imgw.pl/datastore
* https://www.kaggle.com/datasets/krystianadammolenda/imgw-hydro-and-meteo-archive-poland

![https://www.un.org/sites/un2.un.org/files/styles/large-article-image-style-16-9/public/field/image/2023/03/52196025795_06f077377a_c.jpg](https://www.un.org/sites/un2.un.org/files/styles/large-article-image-style-16-9/public/field/image/2023/03/52196025795_06f077377a_c.jpg)


> Source: https://www.un.org/en/un-chronicle/future-weather-climate-and-water-across-generations

## **Here we go!**

In [1]:
import requests, zipfile
import pandas as pd
import numpy as np
from io import BytesIO

---
### **Declare variables**

In the `select_year` and `select_month` variables, set the appropriate time range for which you want to retrieve data.
In the `select_categories` and `select_id_datatype` variables, declare the type of data you want to retrieve.

In [2]:
select_years = ['2020', '2021']
select_month = ['07', '08', '09']

select_categories = ['Hydro', 'Meteo']
select_id_datatype = ['B00020S', 'B00604S']

Meaning of 'Hydro' parameter codes:

`B00020S` Water level (operational)   
`B00050S` Water flow rate (operational)   
`B00014A` Water level (observer)     
`B00101A` Water temperature (observer)   

Meaning of 'Meteo' parameter codes:

`B00300S` Air temperature (official)   
`B00305A` Ground temperature (sensor)   
`B00202A` Wind direction (sensor)   
`B00702A` Average wind speed (sensor)   
`B00703A` Maximum speed (sensor)   
`B00608S` Rainfall total 10 minute sensor   
`B00604S` Daily precipitation total   
`B00606S` Hourly precipitation total   
`B00802A` Relative humidity (sensor)   
`B00714A` Largest wind gust in a 10-minute period from a synoptic station   
`B00910A` Stock of water in snow (observer) 

### **Data download script**

The script is written in such a way that it is not necessary to manually download each archive from https://danepubliczne.imgw.pl/datastore.
The data will be stored under a variable that is the ID of the parameter.

In [3]:
for id_datatype in select_id_datatype:
    globals()[id_datatype] = pd.DataFrame()

for category in select_categories:
    for year in select_years:
        for month in select_month:
            print(f'\nSearching the source: {category}-{year}-{month}:')
        
            try:
                url = f'https://danepubliczne.imgw.pl/datastore/getfiledown/Arch/Telemetria/{category}/{year}/{category}_{year}-{month}.zip'
                request = requests.get(url)
                archive = zipfile.ZipFile(BytesIO(request.content))
            except:
                url = f'https://danepubliczne.imgw.pl/datastore/getfiledown/Arch/Telemetria/{category}/{year}/{category}_{year}-{month}.ZIP'
                request = requests.get(url)
                archive = zipfile.ZipFile(BytesIO(request.content))

            for file in archive.namelist():
                for id_datatype in select_id_datatype:
                    if file.split('_')[0] == id_datatype:
                        print(f' -- File found: {file}')

                        df = pd.read_csv(archive.open(file), sep = ';', decimal = ',', header = None, low_memory = False, usecols = [0, 1, 2, 3])
                        globals()[id_datatype] = pd.concat([globals()[id_datatype], df], ignore_index = True, sort = False)

        archive.close()

for id_datatype in select_id_datatype:
    globals()[id_datatype].columns = ['Measurement point ID', 'Measurement parameter ID', 'Measurement time stamp', 'Measured value']

print(f'\nDownload completed!')
print(f'The data can be accessed under the variables: {select_id_datatype}')


Searching the source: Hydro-2020-07:
 -- File found: B00020S_2020_07.csv

Searching the source: Hydro-2020-08:
 -- File found: B00020S_2020_08.csv

Searching the source: Hydro-2020-09:
 -- File found: B00020S_2020_09.csv

Searching the source: Hydro-2021-07:
 -- File found: B00020S_2021_07.csv

Searching the source: Hydro-2021-08:
 -- File found: B00020S_2021_08.csv

Searching the source: Hydro-2021-09:
 -- File found: B00020S_2021_09.csv

Searching the source: Meteo-2020-07:
 -- File found: B00604S_2020_07.csv

Searching the source: Meteo-2020-08:
 -- File found: B00604S_2020_08.csv

Searching the source: Meteo-2020-09:
 -- File found: B00604S_2020_09.csv

Searching the source: Meteo-2021-07:
 -- File found: B00604S_2021_07.csv

Searching the source: Meteo-2021-08:
 -- File found: B00604S_2021_08.csv

Searching the source: Meteo-2021-09:
 -- File found: B00604S_2021_09.csv

Download completed!
The data can be accessed under the variables: ['B00020S', 'B00604S']


### **Preview of downloaded data**

In [4]:
B00020S

,Measurement point ID,Measurement parameter ID,Measurement time stamp,Measured value
0,150190130,B00020S,2020-07-01 00:00,127.0
1,150190130,B00020S,2020-07-01 00:10,127.0
2,150190130,B00020S,2020-07-01 00:20,127.0
3,150190130,B00020S,2020-07-01 00:30,127.0
4,150190130,B00020S,2020-07-01 00:40,127.0
...,...,...,...,...
15842099,154160060,B00020S,2021-09-26 06:00,78.0
15842100,154160060,B00020S,2021-09-27 06:00,75.0
15842101,154160060,B00020S,2021-09-28 06:00,72.0
15842102,154160060,B00020S,2021-09-29 06:00,74.0


In [5]:
B00604S

,Measurement point ID,Measurement parameter ID,Measurement time stamp,Measured value
0,250180590,B00604S,2020-07-01 06:00,0.0
1,250180590,B00604S,2020-07-02 06:00,0.0
2,250180590,B00604S,2020-07-03 06:00,3.2
3,250180590,B00604S,2020-07-04 06:00,0.0
4,250180590,B00604S,2020-07-05 06:00,0.0
...,...,...,...,...
98126,252150270,B00604S,2021-09-26 06:00,0.0
98127,252150270,B00604S,2021-09-27 06:00,0.0
98128,252150270,B00604S,2021-09-28 06:00,1.0
98129,252150270,B00604S,2021-09-29 06:00,0.0
